# Air Force 1 GEO Similarity Analysis  
## Output Folder Version: `output/airforce/`

This notebook is updated for your current folder structure.

It expects the generated output files to be inside:

```text
output/airforce/
```

Expected output filenames:

```text
output/airforce/nike-air-force-1-07-2023-geo-reference-rewrite.txt
output/airforce/nike-air-force-1-07-2026-current-page-extraction.txt
```

It compares three versions:

1. `2023 filtered original`
2. `2026 current page extraction`
3. `2023 GEO-reference rewrite`

It calculates similarity for exactly three scopes:

1. **Overall**
2. **Product description**
3. **Benefits**

For each scope:

```text
A = similarity(2026 current extraction, 2023 original)
B = similarity(2026 current extraction, 2023 GEO-reference)
Shift = B - A
```

It saves one Excel workbook:

```text
output/airforce/airforce1_overall_description_benefits_similarity.xlsx
```


In [1]:
from pathlib import Path
import re
import difflib
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 1. Project path setup

Run this notebook from the project root folder containing:

```text
filtered_text/
output/airforce/
```

If automatic path detection fails, manually edit `ROOT`.


In [2]:
cwd = Path.cwd()

def looks_like_project_root(path: Path) -> bool:
    return (path / "filtered_text").exists() and (path / "output").exists()

if looks_like_project_root(cwd):
    ROOT = cwd
elif looks_like_project_root(cwd.parent):
    ROOT = cwd.parent
else:
    ROOT = cwd

# Manual override if needed:
# ROOT = Path(r"C:/path/to/GEO_Nike_Product_Page_Detection_Package")

FILTERED_DIR = ROOT / "filtered_text"
OUTPUT_BASE_DIR = ROOT / "output"
AIRFORCE_OUTPUT_DIR = OUTPUT_BASE_DIR / "airforce"
AIRFORCE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXCEL_PATH = AIRFORCE_OUTPUT_DIR / "airforce1_overall_description_benefits_similarity.xlsx"

print("ROOT:", ROOT)
print("FILTERED_DIR exists:", FILTERED_DIR.exists())
print("AIRFORCE_OUTPUT_DIR exists:", AIRFORCE_OUTPUT_DIR.exists())
print("Excel output:", EXCEL_PATH)


ROOT: c:\Users\junse\Documents\research\Finance Research\GEO_Nike_Product_Page_Detection_Package
FILTERED_DIR exists: True
AIRFORCE_OUTPUT_DIR exists: True
Excel output: c:\Users\junse\Documents\research\Finance Research\GEO_Nike_Product_Page_Detection_Package\output\airforce\airforce1_overall_description_benefits_similarity.xlsx


## 2. Load input files

Required filtered files:

```text
filtered_text/2023_Nike_Air_Force_1_filtered_product_specific.txt
filtered_text/2026_Nike_Air_Force_1_filtered_product_specific.txt
```

Required generated output files:

```text
output/airforce/nike-air-force-1-07-2023-geo-reference-rewrite.txt
output/airforce/nike-air-force-1-07-2026-current-page-extraction.txt
```

The notebook uses flexible matching, but it prioritizes these exact hyphenated names.


In [3]:
def find_one(folder: Path, patterns, required=True):
    if isinstance(patterns, str):
        patterns = [patterns]
    matches = []
    for pattern in patterns:
        matches.extend(folder.glob(pattern))
    matches = sorted(set(matches))
    if not matches:
        if required:
            available = [p.name for p in folder.glob("*")] if folder.exists() else "folder does not exist"
            raise FileNotFoundError(
                f"No file found in {folder} for patterns: {patterns}\nAvailable files: {available}"
            )
        return None
    if len(matches) > 1:
        print(f"[Warning] Multiple files matched {patterns}. Using first:")
        for m in matches:
            print(" -", m.name)
    return matches[0]

path_2023_filtered = find_one(FILTERED_DIR, ["*2023*filtered*.txt"])
path_2026_filtered = find_one(FILTERED_DIR, ["*2026*filtered*.txt"])

path_2023_geo = find_one(AIRFORCE_OUTPUT_DIR, [
    "nike-air-force-1-07-2023-geo-reference-rewrite.txt",
    "*2023*geo*reference*rewrite*.txt",
    "*2023*reference*.txt",
])

path_2026_current_extraction = find_one(AIRFORCE_OUTPUT_DIR, [
    "nike-air-force-1-07-2026-current-page-extraction.txt",
    "*2026*current*page*extraction*.txt",
    "*2026*extraction*.txt",
])

def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")

text_2023_original = read_text(path_2023_filtered)
text_2026_filtered = read_text(path_2026_filtered)
text_2023_geo = read_text(path_2023_geo)
text_2026_current = read_text(path_2026_current_extraction)

input_files_df = pd.DataFrame([
    {"role": "2023 filtered original", "path": str(path_2023_filtered)},
    {"role": "2026 filtered current backup", "path": str(path_2026_filtered)},
    {"role": "2023 GEO-reference rewrite", "path": str(path_2023_geo)},
    {"role": "2026 current page extraction", "path": str(path_2026_current_extraction)},
])

input_files_df


,role,path
0,2023 filtered original,c:\Users\junse\Documents\research\Finance Rese...
1,2026 filtered current backup,c:\Users\junse\Documents\research\Finance Rese...
2,2023 GEO-reference rewrite,c:\Users\junse\Documents\research\Finance Rese...
3,2026 current page extraction,c:\Users\junse\Documents\research\Finance Rese...


## 3. Extract only Overall, Product description, and Benefits

The 2026 current extraction and 2023 GEO-reference output should have aligned headings.  
The 2023 filtered original also contains `Product description` and `Benefits`.


In [4]:
TARGET_SCOPES = ["Overall", "Product description", "Benefits"]

SECTION_VARIANTS = {
    "Product description": [
        r"product description",
        r"product overview",
        r"overview",
        r"short shopper-focused summary",
        r"short shopper focused summary",
    ],
    "Benefits": [
        r"benefits",
        r"comfort and cushioning",
        r"materials and construction",
        r"outsole and durability",
        r"style and everyday use",
    ],
}

ALL_POSSIBLE_SECTION_HEADINGS = [
    "Product identity",
    "Key Product Facts",
    "Available sizes",
    "Available Sizes",
    "Size and fit",
    "Size & Fit",
    "Fit Guidance",
    "Payment information",
    "Shipping and pickup",
    "Shipping, returns, and pickup",
    "Shipping, Returns, and Pickup",
    "Shipping, Returns, Pickup, and Payment",
    "Shipping and Returns",
    "Product description",
    "Product Description",
    "Product Overview",
    "Overview",
    "Benefits",
    "Product details",
    "Product Details",
    "Materials and Construction",
    "Comfort and Cushioning",
    "Outsole and Durability",
    "Style and Everyday Use",
    "Air Force 1 origins",
    "Air Force 1 Origins",
    "Reviews and shopper feedback",
    "Reviews and Shopper Feedback",
    "Selected review evidence",
    "AI-generated summary",
    "Short shopper-focused summary",
    "Short Shopper-Focused Summary",
]

def clean_heading(line: str) -> str:
    line = line.strip()
    line = re.sub(r"^#+\s*", "", line)
    line = re.sub(r"[:\-]+$", "", line)
    return line.strip()

def is_heading(line: str) -> bool:
    cleaned = clean_heading(line)
    if not cleaned:
        return False
    return any(cleaned.lower() == h.lower() for h in ALL_POSSIBLE_SECTION_HEADINGS)

def heading_matches(line: str, variants) -> bool:
    cleaned = clean_heading(line).lower()
    return any(re.fullmatch(v, cleaned, flags=re.IGNORECASE) for v in variants)

def extract_section_by_variants(text: str, variants) -> str:
    lines = text.splitlines()
    starts = []
    for i, line in enumerate(lines):
        if heading_matches(line, variants):
            starts.append(i)
    if not starts:
        return ""

    extracted_blocks = []
    for start_idx in starts:
        end_idx = len(lines)
        for j in range(start_idx + 1, len(lines)):
            if is_heading(lines[j]):
                end_idx = j
                break
        block = "\n".join(lines[start_idx + 1:end_idx]).strip()
        if block:
            extracted_blocks.append(block)
    return "\n\n".join(extracted_blocks).strip()

def get_scope_text(text: str, scope: str) -> str:
    if scope == "Overall":
        return text.strip()
    return extract_section_by_variants(text, SECTION_VARIANTS[scope])

scope_texts = {}
scope_rows = []

for scope in TARGET_SCOPES:
    t_2023_original = get_scope_text(text_2023_original, scope)
    t_2026_current = get_scope_text(text_2026_current, scope)
    t_2023_geo = get_scope_text(text_2023_geo, scope)

    scope_texts[scope] = {
        "2023_original": t_2023_original,
        "2026_current": t_2026_current,
        "2023_geo_reference": t_2023_geo,
    }

    scope_rows.append({
        "scope": scope,
        "2023_original_found": bool(t_2023_original.strip()),
        "2026_current_found": bool(t_2026_current.strip()),
        "2023_geo_reference_found": bool(t_2023_geo.strip()),
        "2023_original_words": len(re.findall(r"\b\w+\b", t_2023_original)),
        "2026_current_words": len(re.findall(r"\b\w+\b", t_2026_current)),
        "2023_geo_reference_words": len(re.findall(r"\b\w+\b", t_2023_geo)),
    })

scope_check_df = pd.DataFrame(scope_rows)
scope_check_df


,scope,2023_original_found,2026_current_found,2023_geo_reference_found,2023_original_words,2026_current_words,2023_geo_reference_words
0,Overall,True,True,True,426,376,599
1,Product description,True,True,True,40,38,119
2,Benefits,True,True,True,42,34,46


## 4. Similarity functions

Included metrics:

1. TF-IDF word unigram cosine  
2. TF-IDF word unigram + bigram cosine  
3. TF-IDF character n-gram cosine  
4. Count-vector cosine  
5. Binary count-vector cosine  
6. Token Jaccard  
7. Token Dice  
8. Token overlap coefficient  
9. SequenceMatcher  
10. Containment of 2026 tokens in comparison text  


In [5]:
STOPWORDS = {
    "the", "and", "or", "a", "an", "to", "of", "in", "for", "with", "on", "at",
    "is", "are", "as", "by", "from", "this", "that", "it", "its", "be", "can",
    "has", "have", "when", "where", "into", "not", "only", "source",
}

def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def word_count(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text.lower()))

def token_list(text: str):
    return [
        t for t in re.findall(r"\b[a-zA-Z0-9]+\b", text.lower())
        if t not in STOPWORDS and len(t) > 1
    ]

def token_set(text: str) -> set:
    return set(token_list(text))

def safe_empty(a: str, b: str):
    if not a.strip() and not b.strip():
        return None
    if not a.strip() or not b.strip():
        return 0.0
    return "ok"

def vector_cosine(a: str, b: str, mode: str):
    status = safe_empty(a, b)
    if status is None or status == 0.0:
        return status

    if mode == "tfidf_word_unigram":
        vectorizer = TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 1), min_df=1)
    elif mode == "tfidf_word_unigram_bigram":
        vectorizer = TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), min_df=1)
    elif mode == "tfidf_char_ngram":
        vectorizer = TfidfVectorizer(lowercase=True, analyzer="char_wb", ngram_range=(3, 5), min_df=1)
    elif mode == "count_word_cosine":
        vectorizer = CountVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), min_df=1)
    elif mode == "binary_count_word_cosine":
        vectorizer = CountVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), min_df=1, binary=True)
    else:
        raise ValueError(f"Unknown mode: {mode}")

    matrix = vectorizer.fit_transform([a, b])
    return float(cosine_similarity(matrix[0], matrix[1])[0, 0])

def jaccard(a: str, b: str):
    status = safe_empty(a, b)
    if status is None or status == 0.0:
        return status
    A, B = token_set(a), token_set(b)
    if not A and not B:
        return None
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

def dice(a: str, b: str):
    status = safe_empty(a, b)
    if status is None or status == 0.0:
        return status
    A, B = token_set(a), token_set(b)
    if not A and not B:
        return None
    if not A or not B:
        return 0.0
    return 2 * len(A & B) / (len(A) + len(B))

def overlap_coefficient(a: str, b: str):
    status = safe_empty(a, b)
    if status is None or status == 0.0:
        return status
    A, B = token_set(a), token_set(b)
    if not A and not B:
        return None
    if not A or not B:
        return 0.0
    return len(A & B) / min(len(A), len(B))

def sequence_matcher(a: str, b: str):
    status = safe_empty(a, b)
    if status is None or status == 0.0:
        return status
    return difflib.SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()

def containment_of_first_in_second(first: str, second: str):
    status = safe_empty(first, second)
    if status is None or status == 0.0:
        return status
    A, B = token_set(first), token_set(second)
    if not A:
        return None
    return len(A & B) / len(A)

METRICS = {
    "tfidf_word_unigram_cosine": lambda a, b: vector_cosine(a, b, "tfidf_word_unigram"),
    "tfidf_word_unigram_bigram_cosine": lambda a, b: vector_cosine(a, b, "tfidf_word_unigram_bigram"),
    "tfidf_char_ngram_cosine": lambda a, b: vector_cosine(a, b, "tfidf_char_ngram"),
    "count_word_cosine": lambda a, b: vector_cosine(a, b, "count_word_cosine"),
    "binary_count_word_cosine": lambda a, b: vector_cosine(a, b, "binary_count_word_cosine"),
    "token_jaccard": jaccard,
    "token_dice": dice,
    "token_overlap_coefficient": overlap_coefficient,
    "sequence_matcher": sequence_matcher,
    "containment_of_2026_tokens_in_comparison": containment_of_first_in_second,
}


## 5. Run similarity analysis

For each scope and metric:

```text
A = similarity(2026 current extraction, 2023 original)
B = similarity(2026 current extraction, 2023 GEO-reference)
Shift = B - A
```


In [6]:
rows = []

for scope, texts in scope_texts.items():
    current_2026 = texts["2026_current"]
    original_2023 = texts["2023_original"]
    geo_2023 = texts["2023_geo_reference"]

    for metric_name, metric_func in METRICS.items():
        A = metric_func(current_2026, original_2023)
        B = metric_func(current_2026, geo_2023)
        shift = None if A is None or B is None else B - A

        rows.append({
            "scope": scope,
            "metric": metric_name,
            "A_2026_current_extraction_vs_2023_original": A,
            "B_2026_current_extraction_vs_2023_geo_reference": B,
            "shift_B_minus_A": shift,
            "interpretation": (
                "positive: closer to GEO-reference"
                if shift is not None and shift > 0
                else "negative: closer to 2023 original"
                if shift is not None and shift < 0
                else "tie or not available"
            ),
            "2023_original_words": word_count(original_2023),
            "2026_current_words": word_count(current_2026),
            "2023_geo_reference_words": word_count(geo_2023),
        })

similarity_results_df = pd.DataFrame(rows)
similarity_results_df


,scope,metric,A_2026_current_extraction_vs_2023_original,B_2026_current_extraction_vs_2023_geo_reference,shift_B_minus_A,interpretation,2023_original_words,2026_current_words,2023_geo_reference_words
0,Overall,tfidf_word_unigram_cosine,0.574242,0.549209,-0.025033,negative: closer to 2023 original,426,376,599
1,Overall,tfidf_word_unigram_bigram_cosine,0.482359,0.475879,-0.006480,negative: closer to 2023 original,426,376,599
2,Overall,tfidf_char_ngram_cosine,0.712322,0.688292,-0.024030,negative: closer to 2023 original,426,376,599
3,Overall,count_word_cosine,0.625615,0.598723,-0.026893,negative: closer to 2023 original,426,376,599
4,Overall,binary_count_word_cosine,0.465945,0.477012,0.011067,positive: closer to GEO-reference,426,376,599
5,Overall,token_jaccard,0.347826,0.378151,0.030325,positive: closer to GEO-reference,426,376,599
6,Overall,token_dice,0.516129,0.548780,0.032651,positive: closer to GEO-reference,426,376,599
7,Overall,token_overlap_coefficient,0.536585,0.548780,0.012195,positive: closer to GEO-reference,426,376,599
8,Overall,sequence_matcher,0.271966,0.303404,0.031439,positive: closer to GEO-reference,426,376,599
9,Overall,containment_of_2026_tokens_in_comparison,0.536585,0.548780,0.012195,positive: closer to GEO-reference,426,376,599


## 6. Compact summary

This shows the main metrics only. The Excel file saves all metrics.


In [7]:
main_metrics = [
    "tfidf_word_unigram_bigram_cosine",
    "tfidf_char_ngram_cosine",
    "token_jaccard",
    "sequence_matcher",
]

compact_summary_df = similarity_results_df[
    similarity_results_df["metric"].isin(main_metrics)
].copy()

compact_summary_df


,scope,metric,A_2026_current_extraction_vs_2023_original,B_2026_current_extraction_vs_2023_geo_reference,shift_B_minus_A,interpretation,2023_original_words,2026_current_words,2023_geo_reference_words
1,Overall,tfidf_word_unigram_bigram_cosine,0.482359,0.475879,-0.006480,negative: closer to 2023 original,426,376,599
2,Overall,tfidf_char_ngram_cosine,0.712322,0.688292,-0.024030,negative: closer to 2023 original,426,376,599
5,Overall,token_jaccard,0.347826,0.378151,0.030325,positive: closer to GEO-reference,426,376,599
8,Overall,sequence_matcher,0.271966,0.303404,0.031439,positive: closer to GEO-reference,426,376,599
11,Product description,tfidf_word_unigram_bigram_cosine,0.000000,0.000000,0.000000,tie or not available,40,38,119
12,Product description,tfidf_char_ngram_cosine,0.158946,0.149997,-0.008949,negative: closer to 2023 original,40,38,119
15,Product description,token_jaccard,0.021739,0.000000,-0.021739,negative: closer to 2023 original,40,38,119
18,Product description,sequence_matcher,0.014320,0.021201,0.006882,positive: closer to GEO-reference,40,38,119
21,Benefits,tfidf_word_unigram_bigram_cosine,0.303764,0.260376,-0.043388,negative: closer to 2023 original,42,34,46
22,Benefits,tfidf_char_ngram_cosine,0.434311,0.411619,-0.022692,negative: closer to 2023 original,42,34,46


## 7. Small feature table

In [8]:
features = {
    "payment_or_installment_support": [r"\bklarna\b", r"0% interest", r"\$20/month", r"purchase power"],
    "shipping_checkout_visibility": [r"shipping options", r"checkout"],
    "pickup_store_support": [r"free pickup", r"find a store", r"pick-?up"],
    "explicit_fit_half_size_down": [r"fits large", r"half size down"],
    "material_specificity_leather": [r"leather upper", r"smooth leather"],
    "outsole_traction_specificity": [r"rubber outsole", r"traction", r"pivot circles"],
    "review_summary": [r"what reviewers say", r"praised for", r"some note", r"break-in"],
    "ai_generated_summary": [r"ai-generated summary"],
    "creasing_or_quality_variation": [r"creasing", r"quality may vary"],
}

def has_feature(text: str, patterns: list[str]) -> int:
    text_lower = text.lower()
    return int(any(re.search(pattern, text_lower) for pattern in patterns))

overall_texts = {
    "2023_original": scope_texts["Overall"]["2023_original"],
    "2026_current_extraction": scope_texts["Overall"]["2026_current"],
    "2023_geo_reference": scope_texts["Overall"]["2023_geo_reference"],
}

feature_rows = []
for feature, patterns in features.items():
    row = {"feature": feature}
    for name, text in overall_texts.items():
        row[name] = has_feature(text, patterns)
    row["added_or_newly_visible_in_2026_vs_2023"] = int(
        row["2023_original"] == 0 and row["2026_current_extraction"] == 1
    )
    feature_rows.append(row)

feature_df = pd.DataFrame(feature_rows)
feature_df


,feature,2023_original,2026_current_extraction,2023_geo_reference,added_or_newly_visible_in_2026_vs_2023
0,payment_or_installment_support,0,1,0,1
1,shipping_checkout_visibility,0,1,0,1
2,pickup_store_support,1,1,1,0
3,explicit_fit_half_size_down,1,1,1,0
4,material_specificity_leather,0,1,0,1
5,outsole_traction_specificity,0,1,0,1
6,review_summary,0,1,0,1
7,ai_generated_summary,0,1,1,1
8,creasing_or_quality_variation,0,1,0,1


## 8. Section text preview

Use this sheet to confirm Product description and Benefits were extracted correctly.


In [9]:
section_text_preview_rows = []

for scope, texts in scope_texts.items():
    for name, text in texts.items():
        section_text_preview_rows.append({
            "scope": scope,
            "document": name,
            "word_count": word_count(text),
            "text_preview": normalize_text(text)[:1000],
        })

section_text_preview_df = pd.DataFrame(section_text_preview_rows)
section_text_preview_df


,scope,document,word_count,text_preview
0,Overall,2023_original,426,"Label: Nike Air Force 1 product page, 2023 Way..."
1,Overall,2026_current,376,Product identity Product name: Nike Air Force ...
2,Overall,2023_geo_reference,599,Product identity Nike Air Force 1 '07 is a men...
3,Product description,2023_original,40,The radiance lives on in the Nike Air Force 1 ...
4,Product description,2026_current,38,"Comfortable, durable and timeless—it’s number ..."
5,Product description,2023_geo_reference,119,The Nike Air Force 1 '07 is described as the b...
6,Benefits,2023_original,42,The stitched overlays on the upper add heritag...
7,Benefits,2026_current,34,Leather upper with perforated toe box is breat...
8,Benefits,2023_geo_reference,46,The stitched overlays on the upper add heritag...


## 9. Save Excel workbook

Saved to:

```text
output/airforce/airforce1_overall_description_benefits_similarity.xlsx
```


In [10]:
notes_df = pd.DataFrame({
    "note": [
        "This notebook uses output/airforce as the output folder.",
        "It prioritizes the hyphenated filenames: nike-air-force-1-07-2023-geo-reference-rewrite.txt and nike-air-force-1-07-2026-current-page-extraction.txt.",
        "It compares only three scopes: Overall, Product description, and Benefits.",
        "For each scope, A = similarity(2026 current extraction, 2023 original), B = similarity(2026 current extraction, 2023 GEO-reference), and shift = B - A.",
        "Positive shift means the 2026 current extraction is closer to the GEO-reference under that metric.",
        "Negative shift means the 2026 current extraction is closer to the 2023 original under that metric.",
        "This is exploratory evidence only and is not evidence that Nike intentionally adopted GEO.",
        "Always inspect the SectionTextPreview sheet to confirm that Product description and Benefits were parsed correctly.",
    ]
})

with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
    similarity_results_df.to_excel(writer, sheet_name="AllSimilarityMetrics", index=False)
    compact_summary_df.to_excel(writer, sheet_name="CompactSummary", index=False)
    feature_df.to_excel(writer, sheet_name="FeatureTable", index=False)
    scope_check_df.to_excel(writer, sheet_name="ScopeCheck", index=False)
    section_text_preview_df.to_excel(writer, sheet_name="SectionTextPreview", index=False)
    input_files_df.to_excel(writer, sheet_name="InputFiles", index=False)
    notes_df.to_excel(writer, sheet_name="Notes", index=False)

print("Saved Excel:", EXCEL_PATH)


Saved Excel: c:\Users\junse\Documents\research\Finance Research\GEO_Nike_Product_Page_Detection_Package\output\airforce\airforce1_overall_description_benefits_similarity.xlsx


## 10. Final display

In [11]:
print("Saved Excel:", EXCEL_PATH)
display(compact_summary_df)
display(feature_df)
display(scope_check_df)


Saved Excel: c:\Users\junse\Documents\research\Finance Research\GEO_Nike_Product_Page_Detection_Package\output\airforce\airforce1_overall_description_benefits_similarity.xlsx


,scope,metric,A_2026_current_extraction_vs_2023_original,B_2026_current_extraction_vs_2023_geo_reference,shift_B_minus_A,interpretation,2023_original_words,2026_current_words,2023_geo_reference_words
1,Overall,tfidf_word_unigram_bigram_cosine,0.482359,0.475879,-0.006480,negative: closer to 2023 original,426,376,599
2,Overall,tfidf_char_ngram_cosine,0.712322,0.688292,-0.024030,negative: closer to 2023 original,426,376,599
5,Overall,token_jaccard,0.347826,0.378151,0.030325,positive: closer to GEO-reference,426,376,599
8,Overall,sequence_matcher,0.271966,0.303404,0.031439,positive: closer to GEO-reference,426,376,599
11,Product description,tfidf_word_unigram_bigram_cosine,0.000000,0.000000,0.000000,tie or not available,40,38,119
12,Product description,tfidf_char_ngram_cosine,0.158946,0.149997,-0.008949,negative: closer to 2023 original,40,38,119
15,Product description,token_jaccard,0.021739,0.000000,-0.021739,negative: closer to 2023 original,40,38,119
18,Product description,sequence_matcher,0.014320,0.021201,0.006882,positive: closer to GEO-reference,40,38,119
21,Benefits,tfidf_word_unigram_bigram_cosine,0.303764,0.260376,-0.043388,negative: closer to 2023 original,42,34,46
22,Benefits,tfidf_char_ngram_cosine,0.434311,0.411619,-0.022692,negative: closer to 2023 original,42,34,46


,feature,2023_original,2026_current_extraction,2023_geo_reference,added_or_newly_visible_in_2026_vs_2023
0,payment_or_installment_support,0,1,0,1
1,shipping_checkout_visibility,0,1,0,1
2,pickup_store_support,1,1,1,0
3,explicit_fit_half_size_down,1,1,1,0
4,material_specificity_leather,0,1,0,1
5,outsole_traction_specificity,0,1,0,1
6,review_summary,0,1,0,1
7,ai_generated_summary,0,1,1,1
8,creasing_or_quality_variation,0,1,0,1


,scope,2023_original_found,2026_current_found,2023_geo_reference_found,2023_original_words,2026_current_words,2023_geo_reference_words
0,Overall,True,True,True,426,376,599
1,Product description,True,True,True,40,38,119
2,Benefits,True,True,True,42,34,46
